In [ ]:
# Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#
# Portions of this file consist of AI-generated content. AI-assisted
# content has been reviewed and validated by the authors.

# A digital twin of the SO-101, from first principles

A digital twin is only useful if code cannot tell it from the robot. That
reduces to three claims, each checked below with a measurement:

1. **It moves like the arm** - same command interface, same servo dynamics.
2. **It sees like the arm** - a camera rendered where the real one is mounted.
3. **It feels like the arm** - gripper current that comes from contact.

Then the payoff: the twin console of `./run_pipeline.sh --digital-twin`, with
voice control - you speak, and the workshop's own behaviors drive the twin.

What is simulated: the arm, its servos, the wrist camera and the table
scene (MuJoCo). What is real: everything above the arm transport - your
webcam and microphone, YOLOv26s pose and detection, MediaPipe Hands, the
mapper, Whisper, the Llama intent parser and the behaviors. Everything but
llama-server runs inside this kernel; ROS 2 is not needed.

In [ ]:
import os, sys, pathlib, threading, time

os.environ.setdefault("MUJOCO_GL", "egl")  # headless rendering on the iGPU; set before mujoco loads

PROJECT = (pathlib.Path.cwd() / ".." / ".." / "solution").resolve()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

import cv2, numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from jupyter_ui_poll import ui_events

from common.config import load_config

cfg = load_config()
cfg["robot"]["backend"] = "sim"  # the MuJoCo twin instead of the Feetech bus
cfg["robot"]["use_ros2"] = False  # in-process: this kernel owns the (simulated) bus
cfg["cameras"]["operator"]["use_ros2"] = False  # and reads the webcam itself - no ROS 2 needed
LOAD_THRESHOLD = 120  # Present_Current counts the grip behavior treats as contact

## 0. A body

The twin starts as a physics model: the SO-101's links, joints and
actuators (the open MJCF model), on a table with a few objects. MuJoCo
steps it 500 times per second on a CPU thread, locked to wall-clock time -
so one second of waiting is one second of simulated motion.

In [ ]:
from common.motion import sim_world

world = sim_world(cfg)
m = world.model
print(f"{m.nbody} bodies, {m.njnt} joints, {m.nu} actuators, timestep {world.dt * 1000:.0f} ms")
print("objects:", world.objects)

t0, w0 = world.time, time.perf_counter()
time.sleep(1.0)
print(f"sim time advanced {world.time - t0:.3f} s in {time.perf_counter() - w0:.3f} s of wall time")

## 1. It moves like the arm

`make_arm_backend(cfg)` returns `SimArmClient`, a subclass of the real
`LeRobotArm` in which only the LeRobot `SO101Follower` is swapped for a
simulated bus. Joint clamping, the goal-speed/acceleration register writes
and the current readback all run through the same code as on the rig.

Below the bus sits the part that matters physically: an STS3215 servo does
not jump to a goal. Its firmware accelerates, cruises and decelerates (a
trapezoidal profile), and the twin's servo model does the same. Command one
60° step and measure what comes back.

In [ ]:
from common.motion import LeRobotArm, make_arm_backend

arm = make_arm_backend(cfg)
print(f"{type(arm).__name__} is a LeRobotArm: {isinstance(arm, LeRobotArm)}")
print("at rest:", {k: round(v, 1) for k, v in arm.get_joints().items()})


def step_response(joint, delta, seconds=1.5, hz=100):
    """Send one goal, then sample the joint - what a scope on the servo would show."""
    q0 = arm.get_joints()[joint]
    arm.send_joints({joint: q0 + delta})
    t, q, t0 = [], [], time.perf_counter()
    while (now := time.perf_counter() - t0) < seconds:
        t.append(now)
        q.append(arm.get_joints()[joint])
        time.sleep(1.0 / hz)
    return np.array(t), np.array(q) - q0


t, q = step_response("shoulder_pan", 60.0)
v = np.gradient(q, t)
t95 = t[np.argmax(q >= 0.95 * 60.0)]
print(f"peak speed {v.max():.0f} deg/s, 95% of the step after {t95:.2f} s")

fig, ax = plt.subplots(figsize=(8, 3))
ax.step([0, 0, t[-1]], [0, 60, 60], "k--", lw=1, where="post", label="goal")
ax.plot(t, q, label="position (deg)")
ax.plot(t, v / 10, label="speed (deg/s / 10)")
ax.set_xlabel("seconds after the command")
ax.legend()
plt.show()
arm.go_to_rest()

## 2. It sees like the arm

A camera is a pinhole at a pose. The twin's `wrist_cam` sits where the real
arm camera is mounted - beside the gripper, looking along the jaws - so it
moves with the wrist and sees what is being gripped. A render thread draws
every camera on the iGPU (EGL), separate from the physics thread, and
`SimCamera.read()` returns `(ok, bgr)` exactly like `cv2.VideoCapture`.

Present the open jaws, put the cube between them, and look from both the
wrist and a free 3D view.

In [ ]:
from so101_sim import SimCamera
from so101_sim.render import shared_renderer

PRESENT = {"shoulder_pan": 0.0, "shoulder_lift": -35.0, "elbow_flex": 30.0, "wrist_flex": 10.0, "wrist_roll": 0.0}

wrist = SimCamera(world, "arm")  # the arm camera, on the wrist
view = SimCamera(world, "view")  # a free 3D view, for us (800 x 800, framed in sim.cameras.view)

arm.go_to({**PRESENT, "gripper": 85.0})
time.sleep(1.0)  # let the jaws finish opening
world.stage_object("cube", PRESENT)
_, w_img = wrist.read(fresh=True)
_, v_img = view.read(fresh=True)
print("renderer:", shared_renderer(world).renderer)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, img, title in zip(axes, (w_img, v_img), ("wrist camera", "3D view")):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
plt.show()

## 3. It feels like the arm

The real gripper has no touch sensor. The grip behavior infers contact from
the servo's `Present_Current`: closing on nothing, the motor reaches its
goal and the current falls away; closing on an object, the jaws stall and
the motor pushes against it, so the current stays high.

The twin produces that current from the physics - the actuator effort the
contact actually demands - not from a script. Close the jaws twice, once
empty and once on the cube, and compare against the threshold.

In [ ]:
from common.motion import GRIPPER_CLOSED_FLOOR


def squeeze(with_cube, seconds=2.0, hz=20):
    """Open at PRESENT, (place the cube), close to the floor; return the current trace."""
    arm.go_to({**PRESENT, "gripper": 85.0})
    time.sleep(1.0)
    if with_cube:
        world.stage_object("cube", PRESENT)
    else:
        world.hide("cube")
    arm.send_joints({"gripper": GRIPPER_CLOSED_FLOOR})
    trace = []
    for _ in range(int(seconds * hz)):
        time.sleep(1.0 / hz)
        trace.append(arm.get_gripper_load() or 0.0)
    _, img = wrist.read(fresh=True)
    arm.go_to({"gripper": 85.0})  # let go
    return np.array(trace), img


empty, _ = squeeze(with_cube=False)
held, held_view = squeeze(with_cube=True)
print(f"peak current - empty jaws: {empty.max():.0f}   cube: {held.max():.0f}   (threshold {LOAD_THRESHOLD})")

fig, (ax, ax_img) = plt.subplots(1, 2, figsize=(12, 3.8))
t = np.arange(len(held)) / 20
ax.plot(t, empty, label="empty jaws")
ax.plot(t, held, label="cube between the jaws")
ax.axhline(LOAD_THRESHOLD, color="k", ls="--", lw=1, label="grip threshold")
ax.set_xlabel("seconds after close")
ax.set_ylabel("Present_Current (counts)")
ax.legend()
ax_img.imshow(cv2.cvtColor(held_view, cv2.COLOR_BGR2RGB))
ax_img.set_title("wrist camera while gripping")
ax_img.axis("off")
plt.show()
world.place("cube", world.model.body("cube").pos)  # back on the table for the next section
arm.go_to_rest()

## 4. Close the loop - by voice

If the twin moves, sees and feels like the arm, the workshop's code can drive
it without knowing. Below is the same console as `./run_pipeline.sh
--digital-twin`:

- **top row**: what YOLOv26s-pose sees, what MediaPipe Hands sees (both on
  your webcam, `cameras.operator.device`), and what YOLOv26s detection finds in
  the twin's arm camera;
- **bottom left**: the twin, moving;
- **bottom right**: the voice panel - microphone level, what Whisper heard,
  the intent Llama parsed, the running behavior, the gripper current and
  the joints.

Perception is the rig's: YOLO on the NPU (CPU if the NPU is unavailable - the
provider is in each tile title), MediaPipe on the CPU. Without a webcam a
scripted person stands in, and the cell says so.

The console only watches. The webcam and arm camera are fanned out, so a
running behavior and the console both get every frame. The models are tapped,
so the tiles show what the behavior computed, and the console runs a model
itself only while no behavior is using it.

In [ ]:
from common.fixtures import ScriptedHands, ScriptedPose, SyntheticCamera
from common.motion import mapper_from_config
from common.twin_console import TwinConsole
from integration.dispatch import WorkshopContext
from ros.camera_client import make_camera


def perception():
    """Webcam + the three models; a scripted person if the webcam gives no frames."""
    from models.hands import HandTracker
    from models.vision import ObjectDetector, PoseEstimator

    pose, hands, detector = PoseEstimator(cfg), HandTracker(cfg), ObjectDetector(cfg)
    print(f"YOLO-pose on {pose.provider}, YOLO detect on {detector.provider}, MediaPipe Hands on CPU")
    cam = make_camera(cfg, "operator")  # cameras.operator.use_ros2 is off: a direct OpenCV capture
    if cam.read()[0]:
        print(f"webcam: {cfg['cameras']['operator']['device']}")
        return cam, pose, hands, detector
    cam.close()
    print(f"webcam: no frames from {cfg['cameras']['operator']['device']} (in use by another kernel?) "
          "- a scripted person stands in")
    return SyntheticCamera(), ScriptedPose(), ScriptedHands(), detector


cam, pose, hands, detector = perception()
console = TwinConsole(
    arm,
    operator=cam,
    pose=pose,
    hands=hands,
    armcam=wrist,
    detector=detector,
    view=view,
    load_threshold=LOAD_THRESHOLD,
)
ctx = WorkshopContext(
    cfg=cfg,
    arm=arm,
    camera=console.operator,  # behaviors read the fanned-out webcam
    pose=pose,
    hands=hands,
    mapper=mapper_from_config(cfg),
    headless=True,  # behaviors draw into the console, never their own window
)
ctx.say = console.say  # robot speech lands in the voice panel

The voice path is the pipeline's, and its middle is yours:

1. **Ears** - a microphone thread cuts the audio into utterances (voice
   activity detection: ~0.35 s of silence ends one) and Whisper turns each
   into text on the CPU (`common/speech.py`).
2. **Gauntlet** - every transcript goes through *your* `handle_transcript`:
   the stop fast path, the noise gate, the Llama parser on the iGPU, then the
   registry built by *your* `build_registry`.
3. **Order** - a stop acts the moment it is heard, on the microphone thread,
   by setting the stop event every behavior polls. Any other command waits
   its turn on a dispatcher thread, one behavior at a time - just like the
   pipeline.

Each handler is wrapped only so the panel can show it running and how it
ended. Without a microphone, type into the box under the console instead;
without llama-server, a keyword stand-in routes commands, and the cell says
so.

In [ ]:
import queue

from common.fixtures import offline_parse_text
from common.intents import CommandRegistry, Intent, ParsedCommand, is_noise, is_stop
from common.speech import MicListener, Whisper
from integration.dispatch import build_registry, handle_transcript


def brain():
    """Llama on the iGPU (llama-server, started if needed); the keyword stand-in if it can't come up."""
    try:
        from models.llm import LlamaIntentParser

        llm = LlamaIntentParser(cfg)
        print("intent: Llama 3.2 3B on the iGPU (grammar-constrained)")
        return llm.parse, llm
    except Exception as e:
        print(f"intent: llama-server unavailable ({e}) - the keyword stand-in routes commands")

    def keywords(text):
        d = offline_parse_text(text)
        params = {"object": d["object"]} if d.get("object") else {}
        return ParsedCommand(Intent.from_string(d["command"]), params=params, transcript=text)

    return keywords, None


parse_intent, llm = brain()


def parse(text):
    cmd = parse_intent(text)
    console.parsed(cmd)
    return cmd


behaviors = build_registry()
ctx.registry = CommandRegistry()
for intent in behaviors.intents():
    ctx.registry.add(intent, console.track(intent.value, behaviors.get(intent)))
print("voice commands:", ", ".join(i.value for i in ctx.registry.intents()), "- and stop")

commands = queue.Queue()


def on_transcript(text):
    """Every utterance, spoken or typed: a stop acts now, anything else waits its turn."""
    if is_noise(text):  # a cough, keyboard clicks, silence: shown, never dispatched
        console.ignored(text)
        return
    console.heard(text)
    if is_stop(text):
        console.stopped()
        handle_transcript(ctx, text, parse)  # sets ctx.stop_event: the running behavior ends
    else:
        commands.put(text)


whisper = Whisper(cfg)
mic = MicListener(cfg, whisper.transcribe, on_transcript, on_level=console.on_level).start()
console.transcribing = lambda: mic.transcribing
time.sleep(0.5)
if mic.error:
    console.voice = "typed"
    print(f"microphone: unavailable ({mic.error}) - type commands under the console")
else:
    print(f"microphone: listening, Whisper on {whisper.provider}")

Run the console. Say **"copy my movements"** and move your right arm: the
twin follows your wrist (left/right, up/down), your reach (forward/back),
your wrist bend, the thumb-index angle (roll) and your pinch (gripper). Say
**"wave"**, **"dance"** or **"go home"** for the scripted behaviors, and
**"stop"** at any time. *end session* stops whatever is running and parks
the arm; run the cell again to resume.

In [ ]:
FPS = 15


def jpeg(img):
    return cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 80])[1].tobytes()


def submit(change):
    text = change["new"].strip()
    if text:
        typed.value = ""
        on_transcript(text)


screen = widgets.Image(format="jpeg", layout=widgets.Layout(max_height="90vh"))  # 1200 x 1100, scaled to fit
stop = widgets.Button(description="stop", button_style="danger")
typed = widgets.Text(placeholder="or type a command, then Enter", continuous_update=False,
                     layout=widgets.Layout(width="400px"))
end = widgets.Button(description="end session")
finished = threading.Event()
stop.on_click(lambda _: on_transcript("stop"))
typed.observe(submit, names="value")
end.on_click(lambda _: (finished.set(), ctx.stop_event.set()))
display(widgets.VBox([screen, widgets.HBox([stop, typed, end])]))


def dispatch_loop():
    """Behaviors run here, one at a time, in the order they were asked for."""
    while not finished.is_set():
        try:
            text = commands.get(timeout=0.2)
        except queue.Empty:
            continue
        try:
            handle_transcript(ctx, text, parse)
        except Exception as e:
            console.say(f"(that failed: {e})")


console.say("Ready. Try: 'copy my movements', 'wave', 'dance', 'go home' - or 'stop' at any time.")
dispatcher = threading.Thread(target=dispatch_loop, daemon=True)
dispatcher.start()
with ui_events() as poll:
    while not finished.is_set():
        t0 = time.perf_counter()
        poll(10)  # deliver button clicks and typed commands while this cell runs
        screen.value = jpeg(console.compose())
        time.sleep(max(0.0, 1.0 / FPS - (time.perf_counter() - t0)))
dispatcher.join(timeout=10)
arm.go_to_rest()
print("session ended, arm parked")

## Clean up

On the full system, `python app.py --digital-twin` runs the same twin behind ROS 2:
the node supervisor starts `so101_sim.node` in place of the arm node and the
arm camera node, and the arm and camera clients talk to it over the usual
topics. `./run_pipeline.sh --digital-twin` shows this console in a desktop
window.

In [ ]:
from so101_sim import close_shared_world

finished.set()
ctx.stop_event.set()
dispatcher.join(timeout=5)
mic.stop()
if llm is not None:
    llm.close()  # stops llama-server if this notebook started it
console.close()  # the webcam and arm camera fan-outs, and the cameras they wrap
view.release()
hands.close()
arm.close()
close_shared_world()
print("twin closed")